<a href="https://colab.research.google.com/github/katyayani29verma-cyber/Adaptive-Traffic-Lights/blob/main/Hyperformer_KERAAL_Implementation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/ZhouYuxuanYX/Hyperformer.git
%cd Hyperformer

Cloning into 'Hyperformer'...
remote: Enumerating objects: 271, done.
remote: Counting objects: 100% (57/57), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 271 (delta 48), reused 45 (delta 45), pack-reused 214 (from 1)
Receiving objects: 100% (271/271), 1.03 MiB | 18.21 MiB/s, done.
Resolving deltas: 100% (134/134), done.
/content/Hyperformer


In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected")


CUDA available: True
GPU Name: Tesla T4


In [ ]:
!pip install torch torchvision torchaudio


In [ ]:
!pip install pyyaml tqdm einops tensorboardX


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 4.1 MB/s eta 0:00:00


In [ ]:
!ls


config	     ensemble.sh  graph				  LICENSE  README.md
data	     evaluate.sh  hyperformer.png		  main.py  torchlight
ensemble.py  feeders	  hypergraph_tease_image_new.png  model    train.sh


In [ ]:
!ls config
!ls model
!ls feeders

nturgbd120-cross-set	  nturgbd-cross-subject  ucla
nturgbd120-cross-subject  nturgbd-cross-view
Hyperformer.py	__init__.py
bone_pairs.py  feeder_ntu.py  feeder_ucla.py  __init__.py  tools.py


In [ ]:
!sed -n '1,200p' feeders/feeder_ntu.py


import numpy as np

from torch.utils.data import Dataset

from feeders import tools


class Feeder(Dataset):
    def __init__(self, data_path, label_path=None, p_interval=1, split='train', random_choose=False, random_shift=False,
                 random_move=False, random_rot=False, window_size=-1, normalization=False, debug=False, use_mmap=False,
                 bone=False, vel=False):
        """
        :param data_path:
        :param label_path:
        :param split: training set or test set
        :param random_choose: If true, randomly choose a portion of the input sequence
        :param random_shift: If true, randomly pad zeros at the begining or end of sequence
        :param random_move:
        :param random_rot: rotate skeleton around xyz axis
        :param window_size: The length of the output sequence
        :param normalization: If true, normalize input sequence
        :param debug: If true, only use the first 100 samples
        :param use_mmap: If true, use mmap 

In [3]:
!find / -name "*Brest*" 2>/dev/null

/G1A-Kinect-CTK-R1-Brest-022.txt


In [4]:
import numpy as np

file_path = "/G1A-Kinect-CTK-R1-Brest-022.txt"

data = np.loadtxt(file_path)

print("Shape:", data.shape)

print("\nFirst 10 values of first frame:")
print(data[0][:10])

Shape: (300, 175)

First 10 values of first frame:
[-0.157006 -0.246597  3.413889 -0.00705  -0.021408  0.972109 -0.233443
 -0.170166  0.014198  3.280996]


In [5]:
# reshape raw Kinect data

reshaped = data.reshape(300, 25, 7)

print("New shape:", reshaped.shape)

print("\nFirst joint of first frame:")
print(reshaped[0,0])

New shape: (300, 25, 7)

First joint of first frame:
[-0.157006 -0.246597  3.413889 -0.00705  -0.021408  0.972109 -0.233443]


In [6]:
# keep only xyz coordinates

xyz = reshaped[:, :, :3]

print("XYZ shape:", xyz.shape)

print("\nFirst joint of first frame (xyz only):")
print(xyz[0,0])

XYZ shape: (300, 25, 3)

First joint of first frame (xyz only):
[-0.157006 -0.246597  3.413889]


In [7]:
# convert to Hyperformer format

hyperformer_input = xyz.transpose(2, 0, 1)

# add person dimension (M = 1)
hyperformer_input = np.expand_dims(hyperformer_input, axis=-1)

print("Final shape:", hyperformer_input.shape)

Final shape: (3, 300, 25, 1)


In [11]:
!git clone https://github.com/ZhouYuxuanYX/Hyperformer.git
%cd Hyperformer

Cloning into 'Hyperformer'...
remote: Enumerating objects: 271, done.
remote: Counting objects: 100% (57/57), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 271 (delta 48), reused 45 (delta 45), pack-reused 214 (from 1)
Receiving objects: 100% (271/271), 1.03 MiB | 16.50 MiB/s, done.
Resolving deltas: 100% (134/134), done.
/content/Hyperformer


In [12]:
!ls

config	     ensemble.sh  graph				  LICENSE  README.md
data	     evaluate.sh  hyperformer.png		  main.py  torchlight
ensemble.py  feeders	  hypergraph_tease_image_new.png  model    train.sh


In [13]:
!sed -n '1,120p' model/Hyperformer.py

﻿import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

### torch version too old for timm
### https://github.com/rwightman/pytorch-image-models/blob/master/timm/models/layers
def drop_path(x, drop_prob: float = 0., training: bool = False, scale_by_keep: bool = True):
    """Drop paths (Stochastic Depth) per sample (when applied in main path of residual blocks).
    This is the same as the DropConnect impl I created for EfficientNet, etc networks, however,
    the original name is misleading as 'Drop Connect' is a different form of dropout in a separate paper...
    See discussion: https://github.com/tensorflow/tpu/issues/494#issuecomment-532968956 ... I've opted for
    changing the layer and argument names to 'drop path' rather than mix DropConnect as a layer name and use
    'survival rate' as the argument.
    """
    if drop_prob == 0. or not training:
        return x
    keep_prob = 1 - drop_prob
    shape = (x.shape[0],) + (1,) * (x.n

In [14]:
!grep -n "class" model/Hyperformer.py

27:class DropPath(nn.Module):
99:def import_class(name):
130:    classname = m.__class__.__name__
131:    if classname.find('Conv') != -1:
136:    elif classname.find('BatchNorm') != -1:
142:class TemporalConv(nn.Module):
162:class MultiScale_TemporalConv(nn.Module):
243:class unit_tcn(nn.Module):
259:class MHSA(nn.Module):
370:class Mlp(nn.Module):
402:class unit_vit(nn.Module):
444:class TCN_ViT_unit(nn.Module):
472:class Model(nn.Module):
473:    def __init__(self, num_class=60, num_point=20, num_person=2, graph=None, graph_args=dict(), in_channels=3,
480:            Graph = import_class(graph)
486:        self.num_class = num_class
506:        self.fc = nn.Linear(24*num_of_heads, num_class)
532:        # self.fc = nn.Linear(36 * num_of_heads, num_class)
534:        nn.init.normal_(self.fc.weight, 0, math.sqrt(2. / num_class))


In [15]:
!sed -n '472,560p' model/Hyperformer.py

class Model(nn.Module):
    def __init__(self, num_class=60, num_point=20, num_person=2, graph=None, graph_args=dict(), in_channels=3,
                 drop_out=0, num_of_heads=9, joint_label=[], **kwargs):
        super(Model, self).__init__()

        if graph is None:
            raise ValueError()
        else:
            Graph = import_class(graph)
            self.graph = Graph(**graph_args)

        A = self.graph.A  # 3,25,25

        self.num_of_heads = num_of_heads
        self.num_class = num_class
        self.num_point = num_point
        self.num_person = num_person
        self.data_bn = nn.BatchNorm1d(num_person * in_channels * num_point)
        self.joint_label = joint_label


        self.l1 = TCN_ViT_unit(3, 24*num_of_heads, A, residual=True, num_of_heads=num_of_heads, pe=True, num_point=num_point, layer=1)
        # * num_heads, effect of concatenation following the official implementation
        self.l2 = TCN_ViT_unit(24*num_of_heads, 24*num_of_heads, A, residua

In [16]:
!ls graph

__init__.py  ntu_rgb_d.py  tools.py  ucla.py
